In [2]:
from DWG_to_DuckDB_ETL import *
import os
import shutil
import numpy as np
import math

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\paiva"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\temp"
output_folder = dwg_folder

os.makedirs(temp_folder, exist_ok=True)

filename = "PV Paiva - LVAC - Crossings"

dwg_file = os.path.join(dwg_folder, filename + ".dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, filename + ".dxf")

temp_dwg_file = os.path.join(temp_folder, filename + ".dwg")

conn = load_to_duckdb(dxf_file, "PV Paiva")
print("Data Loaded")
print("DuckDB Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Project PV Paiva already exists
Removed existing data for project: PV Paiva
Successfully loaded project
Loaded 51 entities for project: PV Paiva
Don't forget to close connection.
Data Loaded
DuckDB Loaded


In [5]:
crossings = conn.sql("SELECT type, layer, start_x, start_y, end_x, end_y, length FROM cad_entities WHERE project_name = 'PV Paiva'")
crossings.show()

┌────────────┬──────────────────────────────────┬───────────┬────────────┬───────────┬────────────┬───────────┐
│    type    │              layer               │  start_x  │  start_y   │   end_x   │   end_y    │  length   │
│  varchar   │             varchar              │   float   │   float    │   float   │   float    │   float   │
├────────────┼──────────────────────────────────┼───────────┼────────────┼───────────┼────────────┼───────────┤
│ LWPOLYLINE │ EMF_bt_access_ductbank_07xC      │ 33790.035 │  129006.32 │ 33790.035 │  129002.32 │       4.0 │
│ LWPOLYLINE │ EMF_bt_watercourse_ductbank_04xC │ 33876.465 │  128997.88 │ 33897.367 │  128997.88 │ 20.903572 │
│ LWPOLYLINE │ EMF_bt_access_ductbank_02xC      │ 34337.305 │  129057.78 │ 34337.305 │   129052.8 │  4.988945 │
│ LWPOLYLINE │ EMF_bt_watercourse_ductbank_04xC │ 34366.258 │ 129084.336 │  34379.23 │  129099.56 │      20.0 │
│ LWPOLYLINE │ EMF_bt_watercourse_ductbank_04xC │   34918.0 │ 129396.414 │  34938.66 │ 129396.414 │  20.

In [15]:
boq = conn.execute("""
    SELECT
        layer,
        SUM(length+2) as total_length,
        COUNT(layer) as count
    FROM crossings
    GROUP BY layer
    ORDER BY layer
""").fetch_df()
print(boq)

                               layer  total_length  count
0        EMF_bt_access_ductbank_01xC     47.374074      6
1        EMF_bt_access_ductbank_02xC     61.048230      7
2        EMF_bt_access_ductbank_03xC     48.168860      6
3        EMF_bt_access_ductbank_04xC     17.601588      2
4        EMF_bt_access_ductbank_05xC     42.726798      5
5        EMF_bt_access_ductbank_06xC     17.562185      2
6        EMF_bt_access_ductbank_07xC     36.064722      5
7        EMF_bt_access_ductbank_08xC     35.811016      4
8   EMF_bt_watercourse_ductbank_01xC     96.331860      4
9   EMF_bt_watercourse_ductbank_02xC     22.000000      1
10  EMF_bt_watercourse_ductbank_03xC     69.986544      3
11  EMF_bt_watercourse_ductbank_04xC     89.564383      4
12  EMF_bt_watercourse_ductbank_05xC     79.240501      1
13  EMF_bt_watercourse_ductbank_07xC     22.086281      1
